# 8장. 베이지안 실험설계와 정책 평가## Bayesian Experimental Design and Policy Evaluation### 학습 목표1. **베이즈 정리**와 사전-사후분포 업데이트 과정을 이해한다2. **Normal-Normal 켤레 사전분포**를 활용한 정책 효과 추정을 수행한다3. **Thompson Sampling**을 통한 적응형 실험설계를 구현한다4. **베이즈 팩터(Bayes Factor)**를 계산하고 증거 강도를 평가한다5. **사전분포 민감도 분석**으로 결론의 강건성을 검증한다### 강의 구조| 시간 | 내용 | 유형 ||------|------|------|| Part 1 | 베이지안 추론 기초 | 이론 + 조사 과제 || Part 2 | K대학 사례: 사전→사후 업데이트 | 이론 + 코드 시연 || ☕ 휴식 (15분) | | || Part 3 | Thompson Sampling과 적응형 설계 | 이론 + 조사 과제 || Part 4 | 실습: Thompson Sampling 시뮬레이션 | 코드 작성 || Part 5 | 실습: 베이즈 팩터와 민감도 분석 | 코드 작성 |### 사전 지식- 확률 및 통계 기초 (정규분포, 평균, 분산)- 5장(인과추론 기초)의 처치 효과 개념- Python 기초 (NumPy, Pandas)---_전체 코드는 `practice/chapter08/code/` 참고_

In [ ]:
# ============================================================
# 환경설정 및 라이브러리 로드
# ============================================================
# 처음 사용 시 아래 명령을 터미널에서 먼저 실행하세요:
#   python setup_env.py
# 그 후 VSCode 우측 상단에서 커널을 'AI 기획 강의 (Python 3)'으로 선택하세요.
# ============================================================

import sys

# 패키지 확인
_required = ["numpy", "pandas", "plotly", "scipy"]
_missing = []
for _pkg in _required:
    try:
        __import__(_pkg)
    except ImportError:
        _missing.append(_pkg)
if _missing:
    print(f"❌ 누락된 패키지: {', '.join(_missing)}")
    print("   터미널에서 다음 명령을 실행하세요: python setup_env.py")
    raise SystemExit(1)

# 라이브러리 임포트
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ 라이브러리 로드 완료!")

---# Part 1: 베이지안 추론 기초## 1.1 왜 정책 실험인가?전통적 정책 평가의 한계:- **RCT(무작위 통제 실험)**는 강력하지만, 정책 맥락에서 한계가 있다- 고정 표본 설계(fixed-sample design): 실험 도중 수정 불가- 윤리적 딜레마: 효과 없는 정책에 계속 참여자 배정- p-value의 한계: "효과 없음"을 입증할 수 없음**베이지안 접근법의 장점:**1. 사전 지식(prior)을 명시적으로 반영2. 실험 도중 적응적 수정 가능 (adaptive design)3. 확률적 진술 가능: "정책 효과가 음(-)일 확률은 97%"4. 표본 크기에 자유로움: 데이터가 쌓일수록 업데이트

## 1.2 베이즈 정리 (Bayes' Theorem)$$P(\theta | \text{data}) = \frac{P(\text{data} | \theta) \cdot P(\theta)}{P(\text{data})}$$각 구성요소:- **P(θ)**: 사전분포 (Prior) — 데이터를 보기 전의 믿음- **P(data|θ)**: 우도 (Likelihood) — 모수가 θ일 때 데이터가 관찰될 확률- **P(θ|data)**: 사후분포 (Posterior) — 데이터를 본 후 업데이트된 믿음- **P(data)**: 주변 우도 (Marginal Likelihood) — 정규화 상수핵심 직관:> **사후분포 ∝ 우도 × 사전분포**> > "새로운 믿음 = 데이터의 증거 × 기존 믿음"### Normal-Normal 켤레 사전분포 (Conjugate Prior)사전분포와 우도가 모두 정규분포이면, 사후분포도 정규분포가 된다.- 사전분포: N(μ₀, σ₀²)- 우도 (데이터): x̄ ~ N(θ, σ²/n)  →  간단히 N(μ̂, σ̂²)- **사후분포**: N(μₙ, σₙ²)$$\sigma_n^2 = \frac{1}{\frac{1}{\sigma_0^2} + \frac{1}{\hat{\sigma}^2}}$$$$\mu_n = \sigma_n^2 \left( \frac{\mu_0}{\sigma_0^2} + \frac{\hat{\mu}}{\hat{\sigma}^2} \right)$$직관적 해석:- **사후 정밀도 = 사전 정밀도 + 데이터 정밀도** (정밀도 = 1/분산)- 사후 평균은 사전 평균과 데이터 평균의 **정밀도 가중 평균**- 데이터가 많을수록 → 사후분포가 데이터 쪽으로 이동

In [ ]:
# === 사전분포 → 우도 → 사후분포 업데이트 시각화 ===def normal_posterior(prior_mean, prior_sd, data_mean, data_sd):    """Normal-Normal conjugate posterior 계산"""    prior_var = prior_sd**2    data_var = data_sd**2    post_var = 1 / (1/prior_var + 1/data_var)    post_mean = post_var * (prior_mean/prior_var + data_mean/data_var)    post_sd = np.sqrt(post_var)    return post_mean, post_sd# 예제: 정책 효과 추정prior_mean, prior_sd = 0.0, 2.0        # 약한 사전분포 N(0, 4)data_mean, data_sd = -2.04, 0.55       # 관찰 데이터 (K대학 사례 근사)post_mean, post_sd = normal_posterior(prior_mean, prior_sd, data_mean, data_sd)x = np.linspace(-6, 4, 500)fig = go.Figure()# Priorfig.add_trace(go.Scatter(    x=x, y=stats.norm.pdf(x, prior_mean, prior_sd),    name=f'Prior N({prior_mean}, {prior_sd}²)',    line=dict(color='#E45756', dash='dash', width=2.5),    fill='tozeroy', fillcolor='rgba(228,87,86,0.1)'))# Likelihoodfig.add_trace(go.Scatter(    x=x, y=stats.norm.pdf(x, data_mean, data_sd),    name=f'Likelihood N({data_mean}, {data_sd}²)',    line=dict(color='#4C78A8', dash='dot', width=2.5),    fill='tozeroy', fillcolor='rgba(76,120,168,0.1)'))# Posteriorfig.add_trace(go.Scatter(    x=x, y=stats.norm.pdf(x, post_mean, post_sd),    name=f'Posterior N({post_mean:.2f}, {post_sd:.2f}²)',    line=dict(color='#54A24B', width=3),    fill='tozeroy', fillcolor='rgba(84,162,75,0.15)'))# Zero linefig.add_vline(x=0, line_dash="dash", line_color="gray", line_width=1,              annotation_text="tau=0 (No Effect)", annotation_position="top")fig.update_layout(    title='Bayesian Prior-to-Posterior Update (Normal-Normal Conjugate)',    xaxis_title='Treatment Effect (tau)',    yaxis_title='Density',    legend=dict(x=0.02, y=0.98),    template='plotly_white',    height=500)fig.show()print(f"💡 해석:")print(f"  사전분포: N({prior_mean}, {prior_sd}²) — 효과에 대한 약한 사전 믿음")print(f"  데이터: tau_hat = {data_mean}, SE = {data_sd}")print(f"  사후분포: N({post_mean:.3f}, {post_sd:.3f}²)")print(f"  → 사후분포가 데이터 쪽으로 크게 이동 (데이터 정밀도가 높으므로)")

In [ ]:
# === 다양한 사전분포에 따른 사후분포 비교 ===prior_scenarios = [    {"label": "Strong Skeptic", "mean": 0, "sd": 0.5, "color": "#E45756"},    {"label": "Weak Prior", "mean": 0, "sd": 1.0, "color": "#F58518"},    {"label": "Diffuse Prior", "mean": 0, "sd": 2.0, "color": "#4C78A8"},    {"label": "Very Diffuse", "mean": 0, "sd": 5.0, "color": "#72B7B2"},]data_mean, data_sd = -2.0384, 0.5468  # K대학 SDID 결과x = np.linspace(-5, 3, 500)fig = make_subplots(rows=1, cols=2,                    subplot_titles=('Prior Distributions', 'Posterior Distributions'))results = []for s in prior_scenarios:    # Prior    fig.add_trace(go.Scatter(        x=x, y=stats.norm.pdf(x, s["mean"], s["sd"]),        name=f'{s["label"]} N(0, {s["sd"]}²)',        line=dict(color=s["color"], dash='dash', width=2),        showlegend=True    ), row=1, col=1)        # Posterior    post_mean, post_sd = normal_posterior(s["mean"], s["sd"], data_mean, data_sd)    fig.add_trace(go.Scatter(        x=x, y=stats.norm.pdf(x, post_mean, post_sd),        name=f'Post: N({post_mean:.2f}, {post_sd:.2f}²)',        line=dict(color=s["color"], width=2.5),        showlegend=True    ), row=1, col=2)        # P(tau < 0)    p_negative = stats.norm.cdf(0, post_mean, post_sd)    results.append({        "Prior": s["label"],        "Prior SD": s["sd"],        "Post Mean": round(post_mean, 4),        "Post SD": round(post_sd, 4),        "P(tau<0)": round(p_negative, 4),        "95% CI Lower": round(post_mean - 1.96*post_sd, 4),        "95% CI Upper": round(post_mean + 1.96*post_sd, 4)    })# Data linefor col in [1, 2]:    fig.add_vline(x=0, line_dash="dash", line_color="gray", line_width=1,                  row=1, col=col)fig.update_layout(    title='Effect of Prior Choice on Posterior (K-University Case: tau_hat = -2.038, SE = 0.547)',    template='plotly_white',    height=450)fig.update_xaxes(title_text='Treatment Effect (tau)', row=1, col=1)fig.update_xaxes(title_text='Treatment Effect (tau)', row=1, col=2)fig.update_yaxes(title_text='Density', row=1, col=1)fig.update_yaxes(title_text='Density', row=1, col=2)fig.show()# 결과 테이블result_df = pd.DataFrame(results)print("\n📊 사전분포별 사후 추정 결과:")print(result_df.to_string(index=False))print("\n💡 핵심 발견: 사전분포가 달라도 사후분포는 비슷하게 수렴한다 → 데이터의 증거력이 강함")

---### 📝 이론 과제 8-1 (15분)#### 과제: 빈도주의 vs 베이지안 비교**질문:**1. **베이즈 정리의 핵심 직관**을 자신의 말로 설명하시오. "사전분포", "우도", "사후분포"의 역할을 각각 포함할 것. (3-4문장)2. **빈도주의(Frequentist)와 베이지안(Bayesian) 접근법의 차이**를 다음 기준으로 비교하시오:   - 확률의 정의   - p-value vs 사후확률   - 표본 크기 결정   - 사전 지식 활용3. **정책 실험 맥락에서 베이지안 접근이 유리한 상황** 2가지를 구체적 예시와 함께 설명하시오.**조사 키워드:**- "Bayesian vs Frequentist comparison"- "conjugate prior intuition"- "Bayesian policy evaluation"**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

### ✅ 과제 8-1 제출란**학번:** ___________**이름:** ___________#### 1. 베이즈 정리의 핵심 직관(여기에 작성)#### 2. 빈도주의 vs 베이지안 비교| 기준 | 빈도주의 | 베이지안 ||------|----------|----------|| 확률의 정의 | | || 핵심 지표 | p-value | 사후확률 || 표본 크기 | | || 사전 지식 | | |#### 3. 베이지안이 유리한 정책 실험 상황**상황 1:**(여기에 작성)**상황 2:**(여기에 작성)

---# Part 2: K대학 계열제 사례 — 베이지안 분석## 2.1 K대학 사례 개요K대학교는 2015학년도부터 입학제도를 **학과제 → 계열제**로 전환했다.**핵심 데이터:**- SDID(합성 이중차분법) 추정 처치효과: **tau_hat = −2.0384**- Bootstrap 표준오차: **SE = 0.5468**- 해석: 계열제 전환 후 입시 경쟁률이 약 2.04 감소**베이지안 분석 목표:**1. 사전 믿음을 반영한 처치효과 추정2. "효과가 음(-)일 확률"을 직접 계산3. 사전분포 선택에 따른 결론의 강건성 검증## 2.2 사전분포 설정| 시나리오 | 사전분포 | 의미 ||----------|----------|------|| 강한 회의론 | N(0, 0.5²) | "효과는 거의 0에 가까울 것" || 표준 | N(0, 1²) | "효과 방향도 모름, 크기도 불확실" || 약한 사전정보 | N(0, 2²) | "효과가 있을 수 있으나 방향 불확실" || 매우 확산된 | N(0, 5²) | "거의 무정보 사전분포" |

In [ ]:
# === K대학 사례: 사후분포 상세 분석 ===tau_hat = -2.0384    # SDID 추정치se_hat = 0.5468      # Bootstrap SE# 사전분포 시나리오scenarios = {    "Strong Skeptic N(0, 0.5²)": (0, 0.5),    "Standard N(0, 1²)": (0, 1.0),    "Weak Prior N(0, 2²)": (0, 2.0),    "Very Diffuse N(0, 5²)": (0, 5.0),}x = np.linspace(-5, 3, 1000)colors = ['#E45756', '#F58518', '#4C78A8', '#54A24B']fig = go.Figure()# 데이터 우도 (회색 음영)fig.add_trace(go.Scatter(    x=x, y=stats.norm.pdf(x, tau_hat, se_hat),    name=f'Data Likelihood (tau_hat={tau_hat}, SE={se_hat})',    line=dict(color='gray', width=2, dash='dot'),    fill='tozeroy', fillcolor='rgba(128,128,128,0.1)'))# 각 시나리오별 사후분포print("📊 K대학 계열제 사례: 베이지안 사후분석 결과\n")print(f"{'시나리오':<30} {'사후평균':>10} {'사후SD':>10} {'P(tau<0)':>10} {'95% HDI':>20}")print("-" * 82)for (name, (pm, ps)), color in zip(scenarios.items(), colors):    post_mean, post_sd = normal_posterior(pm, ps, tau_hat, se_hat)    p_neg = stats.norm.cdf(0, post_mean, post_sd)    ci_lo = post_mean - 1.96 * post_sd    ci_hi = post_mean + 1.96 * post_sd        fig.add_trace(go.Scatter(        x=x, y=stats.norm.pdf(x, post_mean, post_sd),        name=f'{name} → Post N({post_mean:.2f}, {post_sd:.2f}²)',        line=dict(color=color, width=2.5)    ))        print(f"{name:<30} {post_mean:>10.4f} {post_sd:>10.4f} {p_neg:>10.4f} [{ci_lo:.3f}, {ci_hi:.3f}]")fig.add_vline(x=0, line_dash="dash", line_color="black", line_width=1,              annotation_text="tau=0", annotation_position="top right")fig.update_layout(    title='K-University Case: Posterior Distributions under Different Priors',    xaxis_title='Treatment Effect (tau)',    yaxis_title='Density',    template='plotly_white',    height=500,    legend=dict(x=0.55, y=0.98, font=dict(size=10)))fig.show()print("\n💡 핵심 발견:")print("  → 모든 사전분포에서 P(tau<0) > 99% → 계열제의 부정적 효과는 매우 강건한 결론")print("  → 사후분포 간 차이가 작음 → 데이터의 증거력이 사전분포를 압도")

In [ ]:
# === 95% 신용구간 (Credible Interval) 비교 ===scenarios_list = [    ("Strong Skeptic\nN(0, 0.5²)", 0, 0.5),    ("Standard\nN(0, 1²)", 0, 1.0),    ("Weak Prior\nN(0, 2²)", 0, 2.0),    ("Very Diffuse\nN(0, 5²)", 0, 5.0),    ("Data Only\n(Flat Prior)", tau_hat, se_hat),  # 사실상 데이터만 반영]fig = go.Figure()for i, (label, pm, ps) in enumerate(scenarios_list):    if label.startswith("Data"):        post_mean, post_sd = tau_hat, se_hat    else:        post_mean, post_sd = normal_posterior(pm, ps, tau_hat, se_hat)        ci_lo = post_mean - 1.96 * post_sd    ci_hi = post_mean + 1.96 * post_sd        # CI bar    fig.add_trace(go.Scatter(        x=[ci_lo, ci_hi], y=[i, i],        mode='lines', line=dict(color='#4C78A8', width=4),        showlegend=False    ))    # Point estimate    fig.add_trace(go.Scatter(        x=[post_mean], y=[i],        mode='markers', marker=dict(color='#E45756', size=12, symbol='diamond'),        showlegend=False,        hovertext=f'Mean: {post_mean:.3f}<br>95% CI: [{ci_lo:.3f}, {ci_hi:.3f}]'    ))# Zero linefig.add_vline(x=0, line_dash="dash", line_color="red", line_width=2,              annotation_text="No Effect (tau=0)", annotation_position="top")fig.update_layout(    title='95% Credible Intervals under Different Priors (K-University)',    xaxis_title='Treatment Effect (tau)',    yaxis=dict(        tickvals=list(range(len(scenarios_list))),        ticktext=[s[0] for s in scenarios_list]    ),    template='plotly_white',    height=400,    showlegend=False)fig.show()print("💡 해석:")print("  → 모든 95% 신용구간이 0을 포함하지 않음")print("  → 사전분포 선택에 관계없이 '계열제가 경쟁률을 하락시켰다'는 결론이 강건함")

---### 📝 이론 과제 8-2 (10분)#### 과제: 사전분포와 신용구간**질문:**1. **신용구간(Credible Interval)**과 **신뢰구간(Confidence Interval)**의 해석 차이를 설명하시오. "95% 신용구간"과 "95% 신뢰구간"이 각각 무엇을 의미하는지 구체적으로 서술할 것. (3-4문장)2. 위 K대학 사례에서 **강한 회의론적 사전분포 N(0, 0.5²)**를 사용했음에도 사후분포가 데이터 쪽으로 크게 이동한 이유를 **정밀도(precision)** 개념으로 설명하시오. (2-3문장)3. 사전분포 선택이 결론에 미치는 영향이 작다는 것은 무엇을 의미하는가? "사전분포 강건성(prior robustness)"의 개념과 정책적 함의를 설명하시오.**조사 키워드:**- "Bayesian credible interval vs confidence interval"- "prior robustness Bayesian analysis"**제출:** 아래 마크다운 셀에 **직접 타이핑**

### ✅ 과제 8-2 제출란**학번:** ___________**이름:** ___________#### 1. 신용구간 vs 신뢰구간(여기에 작성)#### 2. 강한 회의론적 사전분포가 데이터에 의해 압도되는 이유(여기에 작성)#### 3. 사전분포 강건성과 정책적 함의(여기에 작성)

---# ☕ 휴식 시간 (15분)> **지금까지 배운 내용:**> 1. 베이즈 정리와 Normal-Normal 켤레 사전분포> 2. K대학 계열제 사례에서 사전분포→사후분포 업데이트> 3. 사전분포 선택에 대한 결론의 강건성>> **휴식 후 내용:**> - Thompson Sampling을 통한 적응형 실험설계> - 실습: 시뮬레이션과 베이즈 팩터 계산---

---# Part 3: Thompson Sampling과 적응형 실험설계## 3.1 Multi-Armed Bandit 문제**슬롯머신 비유:**- 여러 대의 슬롯머신(arm) 중 가장 보상이 높은 것을 찾아야 한다- **탐색(Exploration)**: 아직 잘 모르는 arm을 시도 → 정보 획득- **활용(Exploitation)**: 현재 최선으로 보이는 arm 선택 → 보상 극대화- **딜레마**: 탐색과 활용 사이의 균형**정책 실험 맥락:**| 슬롯머신 | 정책 실험 ||----------|----------|| Arm | 정책 대안 (예: 계열제, 학과제, 혼합제) || 보상 (Reward) | 정책 성과 (예: 경쟁률 변화) || Exploration | 아직 검증되지 않은 정책 시도 || Exploitation | 현재 최선 정책에 더 많은 참여자 배정 |## 3.2 Thompson Sampling 알고리즘**핵심 아이디어:** 각 arm의 사후분포에서 샘플링하여 자연스럽게 탐색-활용 균형을 달성```반복 t = 1, 2, ..., T:    1. 각 arm k의 사후분포에서 θₖ ~ Posterior_k 샘플링    2. 가장 높은 θₖ를 가진 arm 선택    3. 선택된 arm에서 보상 관찰    4. 해당 arm의 사후분포 업데이트```**장점:**- 불확실한 arm은 분산이 크므로 가끔 높은 값 샘플 → 자연스러운 탐색- 확실히 좋은 arm은 평균이 높으므로 자주 선택 → 효율적 활용- 별도의 하이퍼파라미터 조정 불필요 (ε-Greedy의 ε과 달리)

In [ ]:
# === Thompson Sampling 시뮬레이션 (교수자 시연) ===np.random.seed(42)# 정책 대안 정의 (K대학 맥락)arms = {    'Status Quo (계열제)': {'true_mean': -2.0, 'true_sd': 0.8},    'Restore (학과제)': {'true_mean': 1.5, 'true_sd': 1.0},    'Hybrid (혼합제)': {'true_mean': 0.5, 'true_sd': 0.9},}# 사전분포: N(0, 2²) - 약한 정보prior_mean, prior_sd = 0.0, 2.0n_trials = 100# 초기화posteriors = {arm: {'mu': prior_mean, 'sigma': prior_sd, 'n': 0, 'sum_x': 0}              for arm in arms}# 기록 저장history = {arm: {'counts': [], 'means': [], 'sds': []} for arm in arms}selections = []rewards_list = []cum_reward = 0cum_rewards = []for t in range(1, n_trials + 1):    # 1. Thompson Sampling: 사후분포에서 샘플링    samples = {arm: np.random.normal(p['mu'], p['sigma']) for arm, p in posteriors.items()}        # 2. 최대 샘플 arm 선택    selected = max(samples, key=samples.get)    selections.append(selected)        # 3. 보상 관찰    reward = np.random.normal(arms[selected]['true_mean'], arms[selected]['true_sd'])    rewards_list.append(reward)    cum_reward += reward    cum_rewards.append(cum_reward)        # 4. 사후분포 업데이트 (Normal-Normal, known variance σ²=1)    post = posteriors[selected]    post['n'] += 1    post['sum_x'] += reward    data_var = 1.0    prior_var = prior_sd**2    new_var = 1 / (1/prior_var + post['n']/data_var)    new_mean = new_var * (prior_mean/prior_var + post['sum_x']/data_var)    post['mu'] = new_mean    post['sigma'] = np.sqrt(new_var)        # 기록    for arm in arms:        history[arm]['counts'].append(posteriors[arm]['n'])        history[arm]['means'].append(posteriors[arm]['mu'])        history[arm]['sds'].append(posteriors[arm]['sigma'])# === 결과 시각화 ===arm_colors = {'Status Quo (계열제)': '#E45756', 'Restore (학과제)': '#54A24B', 'Hybrid (혼합제)': '#4C78A8'}fig = make_subplots(rows=2, cols=2,    subplot_titles=(        'Allocation Proportion Over Trials',        'Posterior Mean Convergence',        'Cumulative Reward',        'Final Posterior Distributions'    ))trials = np.arange(1, n_trials + 1)# 1. 할당 비율for arm in arms:    proportions = np.array(history[arm]['counts']) / trials    fig.add_trace(go.Scatter(        x=trials, y=proportions, name=arm,        line=dict(color=arm_colors[arm], width=2),        legendgroup=arm    ), row=1, col=1)# 2. 사후 평균 수렴for arm in arms:    fig.add_trace(go.Scatter(        x=trials, y=history[arm]['means'], name=arm,        line=dict(color=arm_colors[arm], width=2),        showlegend=False, legendgroup=arm    ), row=1, col=2)    # True mean    fig.add_hline(y=arms[arm]['true_mean'], line_dash="dot",                  line_color=arm_colors[arm], line_width=1,                  row=1, col=2)# 3. 누적 보상fig.add_trace(go.Scatter(    x=trials, y=cum_rewards, name='Cumulative Reward',    line=dict(color='#4C78A8', width=2.5),    showlegend=False), row=2, col=1)# 4. 최종 사후분포x_range = np.linspace(-5, 4, 300)for arm in arms:    p = posteriors[arm]    fig.add_trace(go.Scatter(        x=x_range, y=stats.norm.pdf(x_range, p['mu'], p['sigma']),        name=f"{arm} (n={p['n']})",        line=dict(color=arm_colors[arm], width=2),        showlegend=False, legendgroup=arm    ), row=2, col=2)    # True mean marker    fig.add_vline(x=arms[arm]['true_mean'], line_dash="dot",                  line_color=arm_colors[arm], line_width=1,                  row=2, col=2)fig.update_layout(    title='Thompson Sampling: Adaptive Policy Allocation (K-University Scenario)',    template='plotly_white', height=700,    legend=dict(x=1.02, y=1, font=dict(size=10)))fig.update_xaxes(title_text='Trial', row=1, col=1)fig.update_xaxes(title_text='Trial', row=1, col=2)fig.update_xaxes(title_text='Trial', row=2, col=1)fig.update_xaxes(title_text='Treatment Effect (tau)', row=2, col=2)fig.update_yaxes(title_text='Proportion', row=1, col=1)fig.update_yaxes(title_text='Posterior Mean', row=1, col=2)fig.update_yaxes(title_text='Cumulative Reward', row=2, col=1)fig.update_yaxes(title_text='Density', row=2, col=2)fig.show()# 최종 결과 요약print("\n📊 Thompson Sampling 최종 결과:")print(f"{'정책':<25} {'선택 횟수':>10} {'비율':>8} {'사후 평균':>12} {'True Mean':>12}")print("-" * 70)for arm in arms:    p = posteriors[arm]    pct = p['n'] / n_trials * 100    print(f"{arm:<25} {p['n']:>10} {pct:>7.1f}% {p['mu']:>12.3f} {arms[arm]['true_mean']:>12.1f}")print(f"\n누적 보상: {cum_reward:.2f}")print("\n💡 해석: Thompson Sampling이 최선의 정책(학과제)을 빠르게 식별하고 더 많이 배정")

---### 📝 이론 과제 8-3 (10분)#### 과제: 적응형 실험설계의 윤리적 함의**질문:**1. **Thompson Sampling**이 전통적 고정 할당 RCT 대비 **윤리적으로 우월한 이유**를 설명하시오. "탐색-활용 균형(exploration-exploitation)"의 관점에서 서술할 것. (3-4문장)2. K대학 사례에서 Thompson Sampling을 적용했다면, 계열제(현행)에 **배정되는 학생 수가 줄어드는 과정**을 시뮬레이션 결과를 참조하여 설명하시오.3. **적응형 실험설계의 한계점** 1가지를 찾아 설명하시오. (예: 검정력, 추정 편향, 규제 등)**조사 키워드:**- "Thompson Sampling ethics clinical trials"- "adaptive randomization advantages limitations"**제출:** 아래 마크다운 셀에 **직접 타이핑**

### ✅ 과제 8-3 제출란**학번:** ___________**이름:** ___________#### 1. Thompson Sampling의 윤리적 우월성(여기에 작성)#### 2. 계열제 배정 감소 과정 설명(여기에 작성)#### 3. 적응형 실험설계의 한계점(여기에 작성)

---# Part 4: 실습 — Thompson Sampling 구현## 실습 과제 8-4: Thompson Sampling 직접 구현아래 코드의 `TODO` 부분을 완성하여 Thompson Sampling을 구현하시오.**목표:**1. Normal-Normal 사후분포 업데이트 함수 완성2. Thompson Sampling arm 선택 함수 완성  3. 시뮬레이션 실행 및 결과 시각화

In [ ]:
# ========== 실습 과제 8-4: Thompson Sampling 구현 ==========np.random.seed(42)# 정책 대안 (2개로 단순화)policy_arms = {    'Policy A': {'true_mean': -1.5, 'true_sd': 1.0},  # 나쁜 정책    'Policy B': {'true_mean': 1.0, 'true_sd': 1.0},   # 좋은 정책}# TODO 1: 사후분포 업데이트 함수 완성def update_posterior(prior_mu, prior_var, data_sum, n_obs, data_var=1.0):    """    Normal-Normal conjugate posterior update        Args:        prior_mu: 사전분포 평균 (μ₀)        prior_var: 사전분포 분산 (σ₀²)        data_sum: 관찰된 보상의 합계 (Σxᵢ)        n_obs: 관찰 횟수 (n)        data_var: 알려진 데이터 분산 (σ²)        Returns:        tuple: (사후 평균, 사후 분산)        힌트:         post_var = 1 / (1/prior_var + n_obs/data_var)        post_mu = post_var * (prior_mu/prior_var + data_sum/data_var)    """    # ========== 여기서부터 코드를 작성하세요 ==========        # TODO: 사후 분산 계산    post_var = # 여기에 코드 작성        # TODO: 사후 평균 계산    post_mu = # 여기에 코드 작성        # ========== 여기까지 ==========        return post_mu, post_var# TODO 2: Thompson Sampling 선택 함수 완성def thompson_select(posteriors_dict):    """    각 arm의 사후분포에서 샘플링하여 최선의 arm 선택        Args:        posteriors_dict: {arm_name: {'mu': float, 'var': float}}        Returns:        str: 선택된 arm 이름        힌트:        np.random.normal(mu, sqrt(var))로 각 arm에서 샘플        가장 큰 값의 arm 선택    """    # ========== 여기서부터 코드를 작성하세요 ==========        samples = {}    for arm_name, params in posteriors_dict.items():        # TODO: 사후분포에서 샘플링        samples[arm_name] = # 여기에 코드 작성        # TODO: 최대 샘플 값의 arm 선택    selected = # 여기에 코드 작성        # ========== 여기까지 ==========        return selected# TODO 3: 시뮬레이션 실행n_rounds = 80prior_mu, prior_var = 0.0, 4.0  # N(0, 2²)# 각 arm의 상태 초기화arm_state = {arm: {'mu': prior_mu, 'var': prior_var, 'n': 0, 'sum_x': 0}             for arm in policy_arms}selection_history = []reward_history = []for t in range(n_rounds):    # Thompson Sampling으로 arm 선택    posteriors_for_select = {arm: {'mu': s['mu'], 'var': s['var']}                              for arm, s in arm_state.items()}    chosen = thompson_select(posteriors_for_select)        # 보상 관찰    true_params = policy_arms[chosen]    reward = np.random.normal(true_params['true_mean'], true_params['true_sd'])        # 상태 업데이트    arm_state[chosen]['n'] += 1    arm_state[chosen]['sum_x'] += reward    new_mu, new_var = update_posterior(        prior_mu, prior_var,         arm_state[chosen]['sum_x'],         arm_state[chosen]['n']    )    arm_state[chosen]['mu'] = new_mu    arm_state[chosen]['var'] = new_var        selection_history.append(chosen)    reward_history.append(reward)# 결과 출력print("📊 Thompson Sampling 시뮬레이션 결과\n")for arm in policy_arms:    s = arm_state[arm]    print(f"{arm}: 선택 {s['n']}회, 사후평균 = {s['mu']:.3f}, True = {policy_arms[arm]['true_mean']}")print(f"\n누적 보상: {sum(reward_history):.2f}")print(f"Policy B 선택 비율: {selection_history.count('Policy B')/n_rounds*100:.1f}%")

In [ ]:
# === 결과 시각화 (실습 과제 8-4 계속) ===# 위 코드가 정상 실행되었다면, 아래 시각화 코드를 실행fig = make_subplots(rows=1, cols=2,    subplot_titles=('Selection Proportion Over Time', 'Cumulative Reward'))trials = np.arange(1, n_rounds + 1)# 선택 비율 추이for arm in policy_arms:    cumcount = np.cumsum([1 if s == arm else 0 for s in selection_history])    proportions = cumcount / trials    fig.add_trace(go.Scatter(        x=trials, y=proportions, name=arm,        line=dict(width=2.5)    ), row=1, col=1)# 누적 보상fig.add_trace(go.Scatter(    x=trials, y=np.cumsum(reward_history),    name='Cumulative Reward',    line=dict(color='#4C78A8', width=2.5),    showlegend=False), row=1, col=2)fig.update_layout(    title='Thompson Sampling Results (2-Arm Policy Experiment)',    template='plotly_white', height=400)fig.update_xaxes(title_text='Trial', row=1, col=1)fig.update_xaxes(title_text='Trial', row=1, col=2)fig.update_yaxes(title_text='Proportion', row=1, col=1)fig.update_yaxes(title_text='Cumulative Reward', row=1, col=2)fig.show()

---# Part 5: 실습 — 베이즈 팩터와 민감도 분석## 5.1 베이즈 팩터 (Bayes Factor) 개요**베이즈 팩터**는 두 가설 간 증거의 상대적 강도를 측정한다.$$BF_{10} = \frac{P(\text{data} | H_1)}{P(\text{data} | H_0)}$$- **BF₁₀ > 1**: 대립가설(H₁) 지지- **BF₁₀ < 1**: 귀무가설(H₀) 지지### Savage-Dickey 밀도 비율법점 귀무가설 H₀: τ = 0에 대해:$$BF_{01} = \frac{\text{posterior density at } \tau=0}{\text{prior density at } \tau=0}$$$$BF_{10} = \frac{1}{BF_{01}}$$### Kass & Raftery (1995) 증거 강도 기준| BF₁₀ | 증거 강도 ||-------|----------|| < 1 | H₀ 지지 || 1-3 | 약한 증거 || 3-10 | 실질적 증거 || 10-30 | 강한 증거 || 30-100 | 매우 강한 증거 || > 100 | 결정적 증거 |

In [ ]:
# ========== 실습 과제 8-5: 베이즈 팩터 계산 ==========# K대학 사례 데이터tau_hat = -2.0384se_hat = 0.5468# TODO: Savage-Dickey 베이즈 팩터 계산 함수def compute_bayes_factor(tau_hat, se_hat, prior_mean, prior_sd, tau_null=0):    """    Savage-Dickey density ratio로 베이즈 팩터 계산        Args:        tau_hat: 데이터 추정치 (점 추정)        se_hat: 표준오차        prior_mean: 사전분포 평균        prior_sd: 사전분포 표준편차        tau_null: 귀무가설 값 (보통 0)        Returns:        dict: BF10, BF01, 증거 강도        힌트:        1. 사전분포 밀도: stats.norm.pdf(tau_null, prior_mean, prior_sd)        2. 사후분포 계산: normal_posterior 함수 사용        3. 사후분포 밀도: stats.norm.pdf(tau_null, post_mean, post_sd)        4. BF_01 = posterior_density / prior_density        5. BF_10 = 1 / BF_01    """    # ========== 여기서부터 코드를 작성하세요 ==========        # TODO 1: 사전분포에서 tau_null의 밀도    prior_density = # 여기에 코드 작성        # TODO 2: 사후분포 계산    post_mean, post_sd = # 여기에 코드 작성 (normal_posterior 함수 활용)        # TODO 3: 사후분포에서 tau_null의 밀도    posterior_density = # 여기에 코드 작성        # TODO 4: BF 계산    bf_01 = # 여기에 코드 작성    bf_10 = # 여기에 코드 작성        # ========== 여기까지 ==========        # 증거 강도 분류    if bf_10 < 1: strength = "H0 지지"    elif bf_10 < 3: strength = "약한 증거"    elif bf_10 < 10: strength = "실질적 증거"    elif bf_10 < 30: strength = "강한 증거"    elif bf_10 < 100: strength = "매우 강한 증거"    else: strength = "결정적 증거"        return {'BF10': bf_10, 'BF01': bf_01, 'strength': strength,            'post_mean': post_mean, 'post_sd': post_sd}# 테스트: 기본 사전분포 N(0, 1²)result = compute_bayes_factor(tau_hat, se_hat, prior_mean=0, prior_sd=1.0)print(f"BF₁₀ = {result['BF10']:.2f}")print(f"증거 강도: {result['strength']}")

In [ ]:
# === 사전분포 민감도 분석 ===prior_sds = [0.3, 0.5, 1.0, 2.0, 5.0, 10.0]bf_results = []for sd in prior_sds:    r = compute_bayes_factor(tau_hat, se_hat, prior_mean=0, prior_sd=sd)    bf_results.append({        'Prior SD': sd,        'BF10': r['BF10'],        'log(BF10)': np.log(r['BF10']) if r['BF10'] > 0 else 0,        'Strength': r['strength']    })bf_df = pd.DataFrame(bf_results)print("📊 사전분포 민감도 분석 결과:")print(bf_df.to_string(index=False))# 시각화fig = make_subplots(rows=1, cols=2,    subplot_titles=('Bayes Factor by Prior SD', 'log(BF₁₀) by Prior SD'))# BF10 bar chartcolors = ['#E45756' if bf < 1 else '#F58518' if bf < 10 else '#54A24B' if bf < 100 else '#2CA02C'          for bf in bf_df['BF10']]fig.add_trace(go.Bar(    x=[f'sigma={sd}' for sd in bf_df['Prior SD']],    y=bf_df['BF10'],    marker_color=colors,    text=[f'{bf:.1f}' for bf in bf_df['BF10']],    textposition='outside',    name='BF₁₀'), row=1, col=1)fig.add_hline(y=10, line_dash="dash", line_color="orange",              annotation_text="Strong Evidence (BF=10)", row=1, col=1)fig.add_hline(y=100, line_dash="dash", line_color="green",              annotation_text="Decisive (BF=100)", row=1, col=1)# log BF continuousprior_sds_fine = np.linspace(0.1, 12, 200)log_bfs = []for sd in prior_sds_fine:    r = compute_bayes_factor(tau_hat, se_hat, prior_mean=0, prior_sd=sd)    log_bfs.append(np.log(r['BF10']) if r['BF10'] > 0 else 0)fig.add_trace(go.Scatter(    x=prior_sds_fine, y=log_bfs,    line=dict(color='#4C78A8', width=2.5),    name='log(BF₁₀)',    showlegend=False), row=1, col=2)fig.add_hline(y=np.log(10), line_dash="dash", line_color="orange", row=1, col=2)fig.add_hline(y=np.log(100), line_dash="dash", line_color="green", row=1, col=2)fig.add_hline(y=0, line_dash="dot", line_color="red", row=1, col=2)fig.update_layout(    title='Prior Sensitivity Analysis: Bayes Factor (K-University Case)',    template='plotly_white', height=450, showlegend=False)fig.update_xaxes(title_text='Prior SD (sigma_0)', row=1, col=1)fig.update_xaxes(title_text='Prior SD (sigma_0)', row=1, col=2)fig.update_yaxes(title_text='BF₁₀', row=1, col=1)fig.update_yaxes(title_text='log(BF₁₀)', row=1, col=2)fig.show()print("\n💡 해석:")print("  → 사전분포 SD가 커질수록 BF₁₀이 증가 (넓은 사전분포 → 더 강한 증거)")print("  → 모든 사전분포에서 BF₁₀ >> 10 → 계열제 효과에 대한 강한~결정적 증거")

In [ ]:
# === 보너스: Savage-Dickey 밀도 비율 시각화 ===prior_mean, prior_sd = 0, 1.0post_mean, post_sd = normal_posterior(prior_mean, prior_sd, tau_hat, se_hat)x = np.linspace(-5, 4, 1000)fig = go.Figure()# Priorfig.add_trace(go.Scatter(    x=x, y=stats.norm.pdf(x, prior_mean, prior_sd),    name=f'Prior N({prior_mean}, {prior_sd}²)',    line=dict(color='#E45756', dash='dash', width=2),    fill='tozeroy', fillcolor='rgba(228,87,86,0.08)'))# Posteriorfig.add_trace(go.Scatter(    x=x, y=stats.norm.pdf(x, post_mean, post_sd),    name=f'Posterior N({post_mean:.2f}, {post_sd:.2f}²)',    line=dict(color='#4C78A8', width=2.5),    fill='tozeroy', fillcolor='rgba(76,120,168,0.08)'))# Density at tau=0prior_d0 = stats.norm.pdf(0, prior_mean, prior_sd)post_d0 = stats.norm.pdf(0, post_mean, post_sd)fig.add_trace(go.Scatter(    x=[0], y=[prior_d0], mode='markers',    marker=dict(color='#E45756', size=14, symbol='circle'),    name=f'Prior density(0) = {prior_d0:.4f}'))fig.add_trace(go.Scatter(    x=[0], y=[post_d0], mode='markers',    marker=dict(color='#4C78A8', size=14, symbol='square'),    name=f'Posterior density(0) = {post_d0:.6f}'))# Connecting linefig.add_trace(go.Scatter(    x=[0, 0], y=[post_d0, prior_d0],    mode='lines', line=dict(color='gray', width=2, dash='dot'),    showlegend=False))bf_01 = post_d0 / prior_d0bf_10 = 1 / bf_01fig.add_annotation(    x=0.5, y=prior_d0 * 0.7,    text=f'BF₁₀ = {bf_10:.1f}<br>(Decisive Evidence)',    showarrow=False, font=dict(size=13, color='darkred'),    bgcolor='rgba(255,255,255,0.8)', bordercolor='darkred')fig.add_vline(x=0, line_dash="dash", line_color="gray", line_width=1)fig.update_layout(    title='Savage-Dickey Density Ratio: BF₁₀ Visualization',    xaxis_title='Treatment Effect (tau)',    yaxis_title='Density',    template='plotly_white', height=500,    legend=dict(x=0.55, y=0.98, font=dict(size=10)))fig.show()print(f"💡 Savage-Dickey 해석:")print(f"  Prior density at tau=0: {prior_d0:.4f}")print(f"  Posterior density at tau=0: {post_d0:.6f}")print(f"  BF₀₁ = {bf_01:.6f} (사후밀도/사전밀도)")print(f"  BF₁₀ = {bf_10:.1f} (대립가설 지지 강도)")print(f"  → 사후분포에서 tau=0 부근의 밀도가 극도로 낮음 → 효과 없음(H₀)을 강하게 기각")

---## 🎓 강의 마무리 및 핵심 요약### 오늘 배운 내용1. **베이즈 정리**: 사전분포 × 우도 → 사후분포, Normal-Normal 켤레 사전분포의 닫힌 형태 해2. **K대학 사례**: tau_hat = -2.038의 베이지안 분석 → 모든 사전분포에서 P(tau<0) > 99%3. **Thompson Sampling**: 사후분포 샘플링을 통한 자동 탐색-활용 균형, 윤리적 적응형 실험4. **베이즈 팩터**: Savage-Dickey 밀도 비율로 증거 강도 정량화 (BF₁₀ >> 100 → 결정적 증거)5. **사전분포 민감도**: 다양한 사전분포에서도 결론이 일관됨 → 강건한 정책 증거### 핵심 메시지> 베이지안 접근법은 정책 실험에서 세 가지 핵심 이점을 제공한다:> (1) 사전 지식의 명시적 반영, (2) 적응형 실험 설계를 통한 윤리적 실험, > (3) "효과가 음(-)일 확률은 99%"와 같은 직관적 확률 진술. > K대학 사례는 사전분포 선택에 관계없이 일관된 결론을 보여, 증거의 강건성을 입증한다.### 다음 장 예고> **9장. 베이지안 의사결정** — 사후분포를 활용한 최적 의사결정 규칙, > 손실함수(loss function)와 기대손실 최소화, > 정책 결정에서의 베이지안 의사결정 프레임워크### 📚 추가 학습 자료**참고문헌:**- Gelman, A. et al. (2013). *Bayesian Data Analysis* (3rd ed.). CRC Press.- Kass, R. E., & Raftery, A. E. (1995). Bayes Factors. *Journal of the American Statistical Association*, 90(430).- Russo, D. J. et al. (2018). A Tutorial on Thompson Sampling. *Foundations and Trends in Machine Learning*, 11(1).**Python 라이브러리:**- `scipy.stats` — 확률분포 함수- `pymc` — 베이지안 모델링- `arviz` — 베이지안 분석 시각화---_전체 코드는 `practice/chapter08/code/` 참고_